# ALQAC 2026 — Phase 3: Semi-Supervised Pseudo-Label
## 7-signal ensemble + Fused Candidates law + Pseudo-label CBR expansion


---
### Bước 1: Setup & Config

In [ ]:
!pip install -q rank-bm25 scikit-learn numpy sentence-transformers

In [ ]:
import os, json, re, time, itertools
from collections import Counter, defaultdict
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from rank_bm25 import BM25Okapi
from glob import glob
from sentence_transformers import SentenceTransformer

# ===== THAY <TÊN_DATASET> BẰNG TÊN DATASET TRÊN KAGGLE =====
DATA_DIR = "/kaggle/input/datasets/anoro1/alqac-precomputed"
# === HOẶC dòng sau tự động detect (nếu dataset duy nhất trong input):
DATA_DIR = "/kaggle/input/datasets/anoro1/alqac-precomputed"
PRIVATE_TEST_PATH = "%s/ALQAC_private_test.json" % DATA_DIR
PUBLIC_PATH = "%s/ALQAC2026_public_test.json" % DATA_DIR
EVIDENCE_PATH = "%s/evidence_cache.json" % DATA_DIR
CORPUS_PATH = "%s/corpus_law.json" % DATA_DIR
PRECOMPUTED_LAW_PATH = "%s/private_case_and_law_evidence.json" % DATA_DIR
PRIMARY_PATH = "%s/primary_predictions.json" % DATA_DIR
AUDITOR_PATH = "%s/auditor_predictions.json" % DATA_DIR
BGE_EMB_PATH = "%s/bge_embeddings.json" % DATA_DIR
CASE_EMB_PATH = "%s/case_embeddings.json" % DATA_DIR
GRAPH_FEATURES_PATH = "%s/graph_features.json" % DATA_DIR
OUTPUT = "submission_improved.json"
TOP_K_LAWS = 15
VALID_LABELS = {"A_WIN", "PARTIAL_A_WIN", "PARTIAL_B_WIN", "B_WIN"}
LABEL_ORDER = ["A_WIN", "PARTIAL_A_WIN", "PARTIAL_B_WIN", "B_WIN"]


In [ ]:
# ============= LOAD ALL DATA =============
print("\n=== Loading data ===")
t0 = time.time()

with open(PRIVATE_TEST_PATH, encoding="utf-8") as f:
    test_cases = json.load(f)
test_ids = {c["case_id"] for c in test_cases}
print("Private test cases: %d" % len(test_cases))

if os.path.exists(EVIDENCE_PATH):
    with open(EVIDENCE_PATH, encoding="utf-8") as f:
        EVIDENCE_CACHE = json.load(f)
    print("Evidence cache: %d cases" % len(EVIDENCE_CACHE))
else:
    print("WARNING: evidence_cache.json not found")
    EVIDENCE_CACHE = {}

with open(CORPUS_PATH, encoding="utf-8") as f:
    laws = json.load(f)
articles = []
for law in laws:
    for art in law["content"]:
        articles.append({"law_id": law["law_id"], "aid": int(art["aid"]), "text": art["content_Article"]})
aid_to_law = {a["aid"]: {"law_id": a["law_id"], "text": a["text"]} for a in articles}
print("Law corpus: %d laws, %d articles" % (len(laws), len(articles)))

# Pre-computed law evidence
PRECOMPUTED_LAW = {}
if os.path.exists(PRECOMPUTED_LAW_PATH):
    with open(PRECOMPUTED_LAW_PATH) as f:
        raw = json.load(f)
    for entry in raw:
        cid = entry["case_id"]
        law_ev = []
        for aid in entry.get("law_evidence", []):
            info = aid_to_law.get(aid)
            if info:
                law_ev.append({"law_id": info["law_id"], "aid": aid, "text": info["text"]})
        PRECOMPUTED_LAW[cid] = law_ev[:TOP_K_LAWS]
    print("Pre-computed law evidence: %d cases" % len(PRECOMPUTED_LAW))
else:
    print("WARNING: %s not found — fallback to law_aids from primary" % PRECOMPUTED_LAW_PATH)
    with open(PRIMARY_PATH, encoding="utf-8") as f:
        primary_raw = json.load(f)
    for cid, v in primary_raw.items():
        law_aids = v.get("claims", {}).get("law_aids", [])
        law_ev = []
        for aid in law_aids:
            info = aid_to_law.get(aid)
            if info:
                law_ev.append({"law_id": info["law_id"], "aid": aid, "text": info["text"]})
        PRECOMPUTED_LAW[cid] = law_ev[:TOP_K_LAWS]
    print("Fallback law evidence: %d cases" % len(PRECOMPUTED_LAW))

PRIMARY_PREDS = {}
if os.path.exists(PRIMARY_PATH):
    with open(PRIMARY_PATH, encoding="utf-8") as f:
        PRIMARY_PREDS = json.load(f)
    print("Primary predictions: %d cases" % len(PRIMARY_PREDS))
else:
    print("WARNING: primary_predictions.json not found")

AUDITOR_PREDS = {}
if os.path.exists(AUDITOR_PATH):
    with open(AUDITOR_PATH, encoding="utf-8") as f:
        AUDITOR_PREDS = json.load(f)
    print("Auditor predictions: %d cases" % len(AUDITOR_PREDS))
else:
    print("WARNING: auditor_predictions.json not found")

# Embeddings
BGE_EMBEDDINGS = {}
if os.path.exists(BGE_EMB_PATH):
    with open(BGE_EMB_PATH) as f:
        BGE_EMBEDDINGS = json.load(f)
    print("BGE embeddings: %d cases (dim=%d)" % (
        len(BGE_EMBEDDINGS),
        len(next(iter(BGE_EMBEDDINGS.values()))) if BGE_EMBEDDINGS else 0))

CASE_EMBEDDINGS = {}
if os.path.exists(CASE_EMB_PATH):
    with open(CASE_EMB_PATH) as f:
        CASE_EMBEDDINGS = json.load(f)
    print("Case embeddings: %d cases (dim=%d)" % (
        len(CASE_EMBEDDINGS),
        len(next(iter(CASE_EMBEDDINGS.values()))) if CASE_EMBEDDINGS else 0))

GRAPH_FEATURES = {}
if os.path.exists(GRAPH_FEATURES_PATH):
    with open(GRAPH_FEATURES_PATH) as f:
        GRAPH_FEATURES = json.load(f)
    print("Graph features: %d cases" % len(GRAPH_FEATURES))

# ============= Public Cases =============
PUBLIC_CASES = []
if os.path.exists(PUBLIC_PATH):
    with open(PUBLIC_PATH, encoding="utf-8") as f:
        raw = json.load(f)
    PUBLIC_CASES = [{
        "case_id": c["case_id"],
        "case_query": c["case_query"],
        "verdict_label": c["verdict_label"],
        "case_fact": c.get("case_fact") or "",
        "court_reasoning": c.get("court_reasoning") or "",
    } for c in raw]
    print("Public cases: %d" % len(PUBLIC_CASES))
else:
    print("WARNING: public test not found")

print("Total load time: %.1fs" % (time.time()-t0))


In [ ]:
# ===========================================================================
# PHASE 1: Load Judge Passes + Private CBR
# ===========================================================================
# Fallback path: judge_passes from outcome_prediction_runs
JUDGE_BASE = os.path.join(DATA_DIR, "judge_passes")
if not os.path.isdir(JUDGE_BASE):
    print("  WARNING: judge_passes folder not found at %s" % JUDGE_BASE)
    JUDGE_BASE = None

JUDGE_PREDS = {}
if JUDGE_BASE:
    print("Loading judge passes from %s..." % JUDGE_BASE)
    t_j = time.time()
    # Walk recursively — subfolder structure may vary on Kaggle
    for root, dirs, files in os.walk(JUDGE_BASE):
        for fname in files:
            if not fname.endswith(".json"):
                continue
            fpath = os.path.join(root, fname)
            with open(fpath) as f:
                try:
                    data = json.load(f)
                except:
                    continue
            if "case_id" not in data or "pass_name" not in data:
                continue
            cid = data["case_id"]
            pname = data["pass_name"].lower()
            if cid not in JUDGE_PREDS:
                JUDGE_PREDS[cid] = {}
            JUDGE_PREDS[cid][pname] = {
                "label": data.get("direct_label"),
                "confidence": data.get("confidence", "LOW"),
                "key_issue": data.get("key_issue", ""),
            }
    print(f"  Loaded judge passes: {len(JUDGE_PREDS)} cases, {time.time()-t_j:.1f}s")
    if JUDGE_PREDS:
        sample = next(iter(JUDGE_PREDS.values()))
        print(f"  Available passes: {list(sample.keys())}")
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

def private_semantic_cbr(cid, bge_embeds, primary_preds, top_k=3):
    """Find similar PRIVATE cases by BGE cosine similarity.
    Returns (pred, max_similarity, similar_pairs).
    """
    if cid not in bge_embeds:
        return None, 0.0, []
    qv = np.array(bge_embeds[cid], dtype=np.float32)
    sims = []
    for oid, vec in bge_embeds.items():
        if oid == cid:
            continue
        s = cosine_sim(qv, np.array(vec, dtype=np.float32))
        sims.append((oid, s))
    sims.sort(key=lambda x: x[1], reverse=True)
    top = sims[:top_k]
    max_sim = top[0][1] if top else 0.0
    votes = Counter()
    for oid, s in top:
        pred = primary_preds.get(oid, {}).get("prediction")
        if pred:
            votes[pred] += s
    if not votes:
        return None, 0.0, []
    return votes.most_common(1)[0][0], max_sim, [(oid, round(s, 4)) for oid, s in top]

print("=== Phase 1 extra data loaded ===")

def private_semantic_cbr_case(cid, case_embeds, primary_preds, top_k=3):
    """CBR using CASE_EMBEDDINGS (4096-dim) as 8th signal."""
    if cid not in case_embeds:
        return None, 0.0, []
    qv = np.array(case_embeds[cid], dtype=np.float32)
    sims = []
    for oid, vec in case_embeds.items():
        if oid == cid:
            continue
        s = cosine_sim(qv, np.array(vec, dtype=np.float32))
        sims.append((oid, s))
    sims.sort(key=lambda x: x[1], reverse=True)
    top = sims[:top_k]
    max_sim = top[0][1] if top else 0.0
    votes = Counter()
    for oid, s in top:
        pred = primary_preds.get(oid, {}).get("prediction")
        if pred:
            votes[pred] += s
    if not votes:
        return None, 0.0, []
    return votes.most_common(1)[0][0], max_sim, [(oid, round(s, 4)) for oid, s in top]

# --- Load Fused Candidates for Law Retrieval ---
FUSED_CANDIDATES = {}  # case_id -> [{"law_id", "aid", "rrf_score"}]
FUSED_PATH_CANDIDATES = os.path.join(DATA_DIR, "fused_candidates.json")
if os.path.exists(FUSED_PATH_CANDIDATES):
    print("Loading fused candidates from %s..." % FUSED_PATH_CANDIDATES)
    with open(FUSED_PATH_CANDIDATES) as fh:
        fused_raw = json.load(fh)
    for entry in fused_raw:
        cid = entry["case_id"]
        cands = []
        for cand in entry.get("candidates", []):
            cands.append({
                "law_id": cand["law_id"],
                "aid": cand["aid"],
                "score": cand.get("rrf_score", 0.0),
            })
        cands.sort(key=lambda x: x["score"], reverse=True)
        FUSED_CANDIDATES[cid] = cands
    print(f"  Loaded {len(FUSED_CANDIDATES)} cases")

# --- Load Reranker Law Evidence (overrides fused_candidates when available) ---
RERANKER_LAW = {}
RERANKER_LAW_PATH = os.path.join(DATA_DIR, "private_case_and_law_evidence.json")
if os.path.exists(RERANKER_LAW_PATH):
    print("Loading reranker law from %s..." % RERANKER_LAW_PATH)
    with open(RERANKER_LAW_PATH) as fh:
        reranker_raw = json.load(fh)
    for entry in reranker_raw:
        cid = entry["case_id"]
        law_ev = []
        for aid in entry.get("law_evidence", []):
            info = aid_to_law.get(aid)
            if info:
                law_ev.append({"law_id": info["law_id"], "aid": aid})
        RERANKER_LAW[cid] = law_ev
    print("  Loaded %d cases from reranker" % len(RERANKER_LAW))

# --- CBR Law Propagation: inherit articles from similar private cases ---
def build_cbr_law_cache(bge_embeds, reranker_law, top_k=3):
    """For each case, find top_k similar cases by BGE cosine, collect their law articles."""
    case_ids = list(bge_embeds.keys())
    if not case_ids:
        return {}
    vectors = np.array([bge_embeds[cid] for cid in case_ids], dtype=np.float32)
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / (norms + 1e-10)
    sim_matrix = vectors @ vectors.T
    cache = {}
    for i, cid in enumerate(case_ids):
        sims = sim_matrix[i].copy()
        sims[i] = -1.0  # exclude self
        top_idx = np.argsort(sims)[::-1][:top_k]
        neighbor_aids = set()
        for idx in top_idx:
            neighbor = case_ids[idx]
            neighbor_law = reranker_law.get(neighbor, [])
            for le in neighbor_law:
                neighbor_aids.add(le["aid"])
        cache[cid] = neighbor_aids
    print("  Built CBR law cache: %d cases, top_k=%d" % (len(cache), top_k))
    return cache

CBR_LAW_CACHE = build_cbr_law_cache(BGE_EMBEDDINGS, RERANKER_LAW, top_k=3) if BGE_EMBEDDINGS else {}



def get_law_evidence_fused(cid, top_k=25):
    """Law evidence: reranker + CBR propagation + fused fallback."""
    base = []
    if RERANKER_LAW and cid in RERANKER_LAW:
        base = list(RERANKER_LAW[cid])
    elif FUSED_CANDIDATES and cid in FUSED_CANDIDATES:
        cands = FUSED_CANDIDATES[cid]
        base = [{"law_id": c["law_id"], "aid": c["aid"]} for c in cands]
    else:
        base = list(PRECOMPUTED_LAW.get(cid, []))
    # Add CBR-propagated law articles (deduplicated)
    cbr_aids = CBR_LAW_CACHE.get(cid, set())
    base_aids = {le["aid"] for le in base}
    for aid in cbr_aids:
        if aid not in base_aids:
            info = aid_to_law.get(aid)
            if info:
                base.append({"law_id": info["law_id"], "aid": aid})
                base_aids.add(aid)
    return base[:top_k]
    if RERANKER_LAW and cid in RERANKER_LAW:
        return RERANKER_LAW[cid][:top_k]
    if FUSED_CANDIDATES and cid in FUSED_CANDIDATES:
        cands = FUSED_CANDIDATES[cid][:top_k]
        return [{"law_id": c["law_id"], "aid": c["aid"]} for c in cands]
    return PRECOMPUTED_LAW.get(cid, [])


---
## CẢI TIẾN 1: Nâng cấp CBR với Hybrid Search + Weighted Vote

In [ ]:
# ===========================================================================
# TIER 1.2: Hybrid Search (BM25 + Dense) + Weighted Majority Voting
# ===========================================================================

# --- BM25 Index ---
PUBLIC_BM25 = None
if PUBLIC_CASES:
    PUBLIC_BM25_TEXTS = [c["case_query"] + " " + (c.get("case_fact") or "")[:1000] for c in PUBLIC_CASES]
    PUBLIC_BM25 = BM25Okapi([t.lower().split() for t in PUBLIC_BM25_TEXTS])

# NOTE: Embedding files chỉ chứa 60 private cases, không có public cases.
# => Dùng sentence-transformer để encode public case queries thành dense vectors.

PUBLIC_EMB_MODEL = None
PUBLIC_EMBEDDINGS = None

if PUBLIC_CASES:
    print("Loading embedding model BAAI/bge-m3...")
    t_emb = time.time()
    PUBLIC_EMB_MODEL = SentenceTransformer("BAAI/bge-m3", device="cpu")
    pub_queries = [c["case_query"] for c in PUBLIC_CASES]
    PUBLIC_EMBEDDINGS = PUBLIC_EMB_MODEL.encode(
        pub_queries, normalize_embeddings=True, show_progress_bar=False
    )
    PUBLIC_EMBEDDINGS = np.array(PUBLIC_EMBEDDINGS, dtype=np.float32)
    print(f"Embeddings: {len(PUBLIC_EMBEDDINGS)} cases, dim={PUBLIC_EMBEDDINGS.shape[1]}, time={time.time()-t_emb:.1f}s")

def find_similar_public_cases_hybrid(case_query, case_id=None, top_k=5, bge_weight=0.5, bm25_weight=0.5):
    """Hybrid Search: BM25 lexical + (Dense nếu có embedding)."""
    if PUBLIC_BM25 is None or not PUBLIC_CASES:
        return []
    n_public = len(PUBLIC_CASES)
    bm25_raw = PUBLIC_BM25.get_scores(case_query.lower().split())
    bm25_max = bm25_raw.max()
    bm25_norm = bm25_raw / bm25_max if bm25_max > 0 else np.zeros_like(bm25_raw)
    if PUBLIC_EMB_MODEL is not None and PUBLIC_EMBEDDINGS is not None:
        q_emb = PUBLIC_EMB_MODEL.encode([case_query], normalize_embeddings=True, show_progress_bar=False)
        dense_scores = (PUBLIC_EMBEDDINGS @ q_emb.T).flatten()
    else:
        dense_scores = np.zeros(n_public, dtype=np.float32)
    scores = bm25_weight * bm25_norm + bge_weight * dense_scores
    top_idx = np.argsort(scores)[::-1]
    result = []
    for i in top_idx:
        if scores[i] <= 0:
            continue
        if case_id is not None and PUBLIC_CASES[i]["case_id"] == case_id:
            continue
        if len(result) >= top_k:
            break
        result.append({
            "case_id": PUBLIC_CASES[i]["case_id"],
            "verdict_label": PUBLIC_CASES[i]["verdict_label"],
            "score": round(float(scores[i]), 4),
        })
    return result


def predict_by_cbr_weighted(case_query, case_id=None, top_k=5, bge_weight=0.5, bm25_weight=0.5):
    """CBR với Weighted Majority Voting.
    Case càng giống → similarity score càng cao → vote càng nặng.
    """
    similar = find_similar_public_cases_hybrid(case_query, case_id=case_id, top_k=top_k,
                                               bge_weight=bge_weight, bm25_weight=bm25_weight)
    if not similar:
        return None, [], None
    votes = defaultdict(float)
    for s in similar:
        votes[s["verdict_label"]] += s["score"]
    top_label = max(votes, key=votes.get)
    top_score = max(votes.values())
    total = sum(votes.values())
    confidence = top_score / total if total > 0 else 0.0
    return top_label, similar, confidence


---
## CẢI TIẾN 2: Quantitative Extraction thông minh hơn

In [ ]:
# ===========================================================================
# TIER 1.1: Quantitative Extraction cải tiến (sentence segmentation)
# ===========================================================================

AMOUNT_RE = re.compile(r"(\d{1,3}(?:\.\d{3})*|\d+)\s*(đồng|đ|vnd)?", re.IGNORECASE)


def extract_all_amounts(text):
    matches = []
    for m in AMOUNT_RE.finditer(text):
        try:
            matches.append(int(m.group(1).replace(".", "")))
        except:
            pass
    return matches


def split_sentences(text):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', text.replace('\n', ' ')) if len(s.strip()) > 10]


def has_currency(sentence):
    return any(kw in sentence.lower() for kw in ["đồng", "đ", "vnd", "tiền", "triệu", "tỷ", "nghìn"])


def get_quantitative_hint_improved(evidence_texts):
    """Improved quantitative extraction với sentence segmentation."""
    all_text = " ".join(evidence_texts)
    sentences = split_sentences(all_text)
    amounts = extract_all_amounts(all_text)
    if not amounts:
        return ""
    claim_inds = ["yêu cầu", "khởi kiện", "đòi", "nợ gốc", "cho vay",
                  "buộc.*phải trả", "phải thanh toán", "đề nghị.*chấp nhận"]
    accept_inds = ["chấp nhận", "buộc", "xử lý", "tuyên", "có hiệu lực",
                   "phải trả", "chấp nhận một phần"]
    claim_amt, accept_amt = None, None
    for s in sentences:
        s_lower = s.lower()
        amt = None
        for m in AMOUNT_RE.finditer(s):
            try:
                a = int(m.group(1).replace(".", ""))
                if has_currency(s):
                    amt = a
            except:
                pass
        if amt is None:
            continue
        for ind in claim_inds:
            if re.search(ind, s_lower):
                claim_amt = amt
                break
        for ind in accept_inds:
            if re.search(ind, s_lower):
                accept_amt = amt
                break
    claim = claim_amt or (max(amounts) if amounts else None)
    if claim is None:
        return ""
    hint = "[ĐỊNH LƯỢNG] Yêu cầu: %dđ" % claim
    if accept_amt:
        hint += " | Được chấp nhận: %dđ (%.1f%%)" % (accept_amt, accept_amt / claim * 100)
    elif "một phần" in all_text.lower():
        hint += " | Có dấu hiệu chấp nhận một phần"
    return hint


---
## CẢI TIẾN 3: Ensemble + Calibration với Grid Search + Cross-Validation

In [ ]:
# ===========================================================================
# TIER 2.5: Ensemble + Calibration cải tiến (Phase 1: 7 signals)
# ===========================================================================

CALIBRATION_MODEL = None
CALIBRATION_SCALER = None

CONF_MAP = {"HIGH": 1.5, "MEDIUM": 1.0, "LOW": 0.6}
JUDGE_CONF_MAP = {"HIGH": 1.5, "MEDIUM": 1.0, "LOW": 0.6}


def resolve_prediction(pred, probs, default="A_WIN"):
    if pred is not None:
        return pred
    if probs:
        return max(probs, key=probs.get)
    return default


def ensemble_predict_7signals(primary_pred, primary_probs, primary_conf,
                               auditor_pred, auditor_conf,
                               cbr_pred, cbr_weight,
                               priv_cbr_pred, priv_cbr_weight,
                               case_emb_pred, case_emb_weight,
                               judge_ad_pred, judge_ad_conf,
                               judge_el_pred, judge_el_conf,
                               judge_re_pred, judge_re_conf,
                               weights):
    """8-signal ensemble: Primary + Auditor + CBR pub/priv + CaseEmb + 3 Judge passes."""
    w = weights
    votes = defaultdict(float)
    # 1. Primary (confidence-scaled)
    pp = resolve_prediction(primary_pred, primary_probs)
    primary_scale = CONF_MAP.get(primary_conf, 1.0)
    votes[pp] += primary_scale * w.get("weight_primary", 1.0)
    # 2. Auditor
    if auditor_pred:
        votes[auditor_pred] += CONF_MAP.get(auditor_conf, 0.8) * w.get("weight_auditor", 0.8)
    # 3. CBR public
    if cbr_pred:
        votes[cbr_pred] += cbr_weight * w.get("weight_cbr", 0.5)
    # 4. CBR private (BGE)
    if priv_cbr_pred:
        votes[priv_cbr_pred] += priv_cbr_weight * w.get("weight_priv_cbr", 0.8)
    # 5. CBR case embeddings (4096d)
    if case_emb_pred:
        votes[case_emb_pred] += case_emb_weight * w.get("weight_case_emb", 0.5)
    # 6-8. Judge passes with agreement boost
    judge_agreement = 0
    judge_labels = []
    if judge_ad_pred:
        judge_labels.append(judge_ad_pred)
    if judge_el_pred:
        judge_labels.append(judge_el_pred)
    if judge_re_pred:
        judge_labels.append(judge_re_pred)
    if judge_labels:
        from collections import Counter as _JCount
        judge_vote_counts = _JCount(judge_labels).most_common()
        if judge_vote_counts:
            judge_agreement = judge_vote_counts[0][1]  # max agreements
    boost = 1.5 if judge_agreement >= 2 else 0.8 if judge_agreement <= 1 else 1.0
    # 6. Judge Adversarial
    if judge_ad_pred:
        votes[judge_ad_pred] += boost * JUDGE_CONF_MAP.get(judge_ad_conf, 0.8) * w.get("weight_judge_ad", 0.6)
    # 7. Judge Elements_Burden
    if judge_el_pred:
        votes[judge_el_pred] += boost * JUDGE_CONF_MAP.get(judge_el_conf, 0.8) * w.get("weight_judge_el", 0.6)
    # 8. Judge Remedy_Outcome
    if judge_re_pred:
        votes[judge_re_pred] += boost * JUDGE_CONF_MAP.get(judge_re_conf, 0.8) * w.get("weight_judge_re", 0.6)
    return max(votes, key=votes.get), dict(votes)


def get_public_simulated_preds(cbr_pred):
    base_pred = cbr_pred or "A_WIN"
    base_probs = {l: 0.5 if l == base_pred else 0.0 for l in LABEL_ORDER}
    return base_pred, base_probs, "MEDIUM"


def build_cbr_cache(public_cases):
    cache = {}
    print("Building CBR cache for %d public cases..." % len(public_cases), end=" ")
    for pc in public_cases:
        cid = pc["case_id"]
        cbr_pred, cbr_sim, _ = predict_by_cbr_weighted(pc["case_query"], case_id=cid)
        cbr_score = cbr_sim[0]["score"] if cbr_sim else None
        cache[cid] = {"pred": cbr_pred, "score": cbr_score}
    print("done (%d cases)" % len(cache))
    return cache


def grid_search_7signals(public_cases, auditor_preds, cbr_cache, judge_preds,
                         weight_ranges=None, top_k=5):
    """Grid Search for 7-signal ensemble weights."""
    if weight_ranges is None:
        weight_ranges = {
            "weight_primary": [0.5, 0.8, 1.0, 1.2, 1.5],
            "weight_auditor": [0.3, 0.5, 0.8, 1.0],
            "weight_cbr": [0.3, 0.5, 0.7],
            "weight_priv_cbr": [0.3, 0.5, 0.8, 1.0],
            "weight_judge_ad": [0.3, 0.5, 0.8],
            "weight_judge_el": [0.3, 0.5, 0.8],
            "weight_judge_re": [0.3, 0.5, 0.8],
            "weight_case_emb": [0.1, 0.3, 0.5, 0.8],
        }
    configs = list(itertools.product(*weight_ranges.values()))
    key_names = list(weight_ranges.keys())
    print("Grid search 7-signal: %d configs" % len(configs))

    scored = []
    for values in configs:
        w = dict(zip(key_names, values))
        correct = 0
        for pc in public_cases:
            cid = pc["case_id"]
            cbr = cbr_cache[cid]
            base_pred, base_probs, base_conf = get_public_simulated_preds(cbr["pred"])
            aud = auditor_preds.get(cid, {})
            cbr_w = 0.5 if cbr["pred"] else 0.0
            # Judge passes for this public case
            jp = judge_preds.get(cid, {})
            ad = jp.get("adversarial", {})
            el = jp.get("elements_burden", {})
            re = jp.get("remedy_outcome", {})
            # Private CBR not available for public cases (no BGE embeds for public)
            priv_pred, priv_w = None, 0.0
            pred, _ = ensemble_predict_7signals(
                base_pred, base_probs, base_conf,
                aud.get("prediction"), aud.get("confidence", "LOW"),
                cbr["pred"], cbr_w,
                priv_pred, priv_w,
                None, 0.0,  # case_emb not available for public
                ad.get("label"), ad.get("confidence", "LOW"),
                el.get("label"), el.get("confidence", "LOW"),
                re.get("label"), re.get("confidence", "LOW"),
                w)
            if pred == pc["verdict_label"]:
                correct += 1
        acc = correct / len(public_cases)
        scored.append((acc, w))

    scored.sort(key=lambda x: x[0], reverse=True)
    top_k = min(top_k, len(scored))
    top_configs = [scored[i][1] for i in range(top_k)]
    print("Top-%d configs:" % top_k)
    for i in range(top_k):
        print("  [%d] acc=%.4f %s" % (i+1, scored[i][0], scored[i][1]))
    return top_configs


# --- Keep calibration functions unchanged ---

def extract_features_improved(primary_probs, primary_pred, auditor_conf,
                              cbr_pred, cbr_agree, cbr_score,
                              has_amounts, primary_confidence=None):
    feats = []
    for lbl in LABEL_ORDER:
        feats.append(primary_probs.get(lbl, 0.0) if primary_probs else 0.0)
    feats.append(CONF_MAP.get(auditor_conf, 0.5) if auditor_conf else 0.5)
    feats.append(cbr_score if cbr_score is not None else 0.0)
    feats.append(1.0 if cbr_agree else 0.0)
    if primary_probs:
        sorted_probs = sorted(primary_probs.values(), reverse=True)
        margin = sorted_probs[0] - (sorted_probs[1] if len(sorted_probs) > 1 else 0.0)
    else:
        margin = 0.0
    feats.append(margin)
    feats.append(1.0 if has_amounts else 0.0)
    feats.append(CONF_MAP.get(primary_confidence, 1.0) if primary_confidence else 1.0)
    return feats


def train_calibration_cv(public_data, public_labels, model_type='rf'):
    global CALIBRATION_MODEL, CALIBRATION_SCALER
    X = np.array([d["features"] for d in public_data])
    y = np.array([LABEL_ORDER.index(l) for l in public_labels])
    if len(X) < 10 or len(set(y)) < 2:
        print("  Insufficient data (%d), skipping" % len(X))
        return
    CALIBRATION_SCALER = StandardScaler()
    X_scaled = CALIBRATION_SCALER.fit_transform(X)
    if model_type == 'rf':
        CALIBRATION_MODEL = RandomForestClassifier(
            n_estimators=200, max_depth=5, min_samples_leaf=2,
            class_weight='balanced', random_state=42)
    elif model_type == 'gb':
        CALIBRATION_MODEL = GradientBoostingClassifier(
            n_estimators=100, max_depth=3, learning_rate=0.1, random_state=42)
    else:
        CALIBRATION_MODEL = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
    try:
        cv = StratifiedKFold(n_splits=min(5, min(Counter(y).values())), shuffle=True, random_state=42)
        cv_scores = cross_val_score(CALIBRATION_MODEL, X_scaled, y, cv=cv, scoring='accuracy')
        print("  CV scores: %s, mean=%.4f, std=%.4f" % (cv_scores, cv_scores.mean(), cv_scores.std()))
    except Exception as e:
        print("  CV failed: %s" % e)
    CALIBRATION_MODEL.fit(X_scaled, y)
    train_acc = (CALIBRATION_MODEL.predict(X_scaled) == y).mean()
    print("  Trained: %d samples, %d classes, train acc=%.3f" % (len(X), len(set(y)), train_acc))


def apply_calibration_improved(features, threshold=0.5):
    if CALIBRATION_MODEL is None or CALIBRATION_SCALER is None:
        return None
    X = np.array([features])
    X_scaled = CALIBRATION_SCALER.transform(X)
    probs = CALIBRATION_MODEL.predict_proba(X_scaled)[0]
    if max(probs) > threshold:
        return LABEL_ORDER[CALIBRATION_MODEL.predict(X_scaled)[0]]
    return None


---
## TRAINING: Grid Search + Calibration trên Public Data

**Lưu ý:** Primary predictions chỉ có cho 60 private cases, không có cho public cases.
Do đó, trên public set ta simulate primary predictions từ CBR (giống baseline).

In [ ]:
# ===========================================================================
# Bước 1: Precompute CBR + Grid Search 7-signal
# ===========================================================================
print("\n=== Building CBR Cache + 7-Signal Grid Search ===\n")
CBR_CACHE = build_cbr_cache(PUBLIC_CASES)
TOP_K_WEIGHTS = grid_search_7signals(PUBLIC_CASES, AUDITOR_PREDS, CBR_CACHE, JUDGE_PREDS, top_k=5)
print()


In [ ]:
# ===========================================================================
# Bước 2: Training Calibration Model với Cross-Validation
# ===========================================================================
print("\n=== Calibration Training (Tier 2.5) ===")
cal_data, cal_labels = [], []

for pc in PUBLIC_CASES:
    cid = pc["case_id"]
    proxy_ev = [(pc.get("case_fact") or "")[:2000], (pc.get("court_reasoning") or "")[:2000]]
    proxy_text = " ".join(proxy_ev)

    cbr = CBR_CACHE[cid]
    cbr_pred, cbr_score = cbr["pred"], cbr["score"]
    has_amt = bool(extract_all_amounts(proxy_text))

    base_pred, base_probs, _ = get_public_simulated_preds(cbr_pred)

    aud = AUDITOR_PREDS.get(cid, {})
    cbr_agree = (cbr_pred == base_pred) if cbr_pred else False

    feats = extract_features_improved(
        base_probs, base_pred, aud.get("confidence", "MEDIUM"),
        cbr_pred, cbr_agree, cbr_score, has_amt,
        primary_confidence="MEDIUM")
    cal_data.append({"features": feats})
    cal_labels.append(pc["verdict_label"])

print("\n--- RandomForest ---")
train_calibration_cv(cal_data, cal_labels, model_type='rf')
print("\n--- GradientBoosting ---")
train_calibration_cv(cal_data, cal_labels, model_type='gb')
print("\n=> Using RandomForest for final prediction")
train_calibration_cv(cal_data, cal_labels, model_type='rf')
print()


In [ ]:
# ===========================================================================
# PHASE 3: Semi-Supervised Pseudo-Labeling
# ===========================================================================
# Pass 1: Predict all 60 private cases, identify high-confidence ones
# Pass 2: Add pseudo-labeled cases to CBR base, re-predict low-confidence cases

HIGH_CONF_CASES = {}  # case_id -> prediction (from pass 1)
LOW_CONF_CASES = {}   # case_id -> {} (to be re-predicted)
UNUSED = None  # threshold inlined below

print("\n=== Phase 3: Pseudo-Labeling Pass 1 ===\n")
t_p3 = time.time()

# First pass: predict all, collect agreement stats
pass1_results = {}
for c in test_cases:
    cid = c["case_id"]; cq = c["case_query"]
    ev_entry = EVIDENCE_CACHE.get(cid, {})
    chunk_texts = ev_entry.get("chunk_texts", [])

    cbr_pred, cbr_similar, _ = predict_by_cbr_weighted(cq, case_id=cid)
    cbr_score = cbr_similar[0]["score"] if cbr_similar else None
    cbr_weight = cbr_score if cbr_score is not None else 0.5

    priv_cbr_pred, priv_cbr_maxsim, _ = private_semantic_cbr(cid, BGE_EMBEDDINGS, PRIMARY_PREDS, top_k=3)
    priv_cbr_weight = priv_cbr_maxsim if priv_cbr_maxsim > 0 else 0.0

    jp = JUDGE_PREDS.get(cid, {})
    j_ad = jp.get("adversarial", {})
    j_el = jp.get("elements_burden", {})
    j_re = jp.get("remedy_outcome", {})

    prim = PRIMARY_PREDS.get(cid, {})
    primary_pred = prim.get("prediction")
    primary_probs = prim.get("probabilities", {})
    aud = AUDITOR_PREDS.get(cid, {})
    auditor_pred = aud.get("prediction")
    auditor_conf = aud.get("confidence", "LOW")

    # Collect all 7 signal votes independently
    signal_labels = {}
    signal_labels["primary"] = resolve_prediction(primary_pred, primary_probs)
    signal_labels["auditor"] = auditor_pred
    signal_labels["cbr_public"] = cbr_pred
    signal_labels["cbr_private"] = priv_cbr_pred
    signal_labels["judge_ad"] = j_ad.get("label")
    signal_labels["judge_el"] = j_el.get("label")
    signal_labels["judge_re"] = j_re.get("label")
    active_signals = [l for l in signal_labels.values() if l is not None]
    if not active_signals:
        active_signals = [resolve_prediction(primary_pred, primary_probs)]
    signal_counts = Counter(active_signals)
    pred = signal_counts.most_common(1)[0][0]
    max_votes = signal_counts.most_common(1)[0][1]
    total_signals = len(active_signals)
    agreement = max_votes / total_signals if total_signals > 0 else 0.0

    prim_conf = prim.get("confidence", "LOW")
    is_high_conf = (max_votes >= 5) or (prim_conf == "HIGH" and max_votes >= 4)

    if is_high_conf and pred in VALID_LABELS:
        HIGH_CONF_CASES[cid] = pred
    else:
        LOW_CONF_CASES[cid] = {
            "pred": pred,
            "agreement": agreement,
            "prim_conf": prim_conf,
            "signal_counts": dict(signal_counts),
        }

    pass1_results[cid] = pred
print(f"Pass 1: {len(HIGH_CONF_CASES)} high-confidence, {len(LOW_CONF_CASES)} low-confidence cases")
print(f"Time: {time.time()-t_p3:.1f}s")

# --- Build expanded CBR base ---
if HIGH_CONF_CASES:
    print(f"\n=== Expanding CBR base with {len(HIGH_CONF_CASES)} pseudo-labeled cases ===")
    expanded_cases = list(PUBLIC_CASES)  # Start with 50 public
    for c in test_cases:
        cid = c["case_id"]
        if cid in HIGH_CONF_CASES:
            expanded_cases.append({
                "case_id": cid,
                "case_query": c["case_query"],
                "case_fact": c.get("case_fact", ""),
                "verdict_label": HIGH_CONF_CASES[cid],
            })
    print(f"Expanded CBR base: {len(expanded_cases)} cases ({len(PUBLIC_CASES)} public + {len(HIGH_CONF_CASES)} pseudo)")

    # Rebuild BM25 index (mất thêm time nhưng chỉ 1 lần)
    EXPANDED_BM25_TEXTS = [c["case_query"] + " " + (c.get("case_fact") or "")[:1000] for c in expanded_cases]
    EXPANDED_BM25 = BM25Okapi([t.lower().split() for t in EXPANDED_BM25_TEXTS])

    # Rebuild expanded embedding cache (public + pseudo)
    EXPANDED_QUERIES = [c["case_query"] for c in expanded_cases]
    EXPANDED_EMBEDDINGS = PUBLIC_EMB_MODEL.encode(
        EXPANDED_QUERIES, normalize_embeddings=True, show_progress_bar=False
    )
    EXPANDED_EMBEDDINGS = np.array(EXPANDED_EMBEDDINGS, dtype=np.float32)
    print(f"Expanded embeddings: {len(EXPANDED_EMBEDDINGS)} cases, dim={EXPANDED_EMBEDDINGS.shape[1]}")

    # Helper: hybrid search on expanded set
    def find_similar_expanded(case_query, case_id=None, top_k=5, bge_weight=0.5, bm25_weight=0.5):
        bm25_raw = EXPANDED_BM25.get_scores(case_query.lower().split())
        bm25_max = bm25_raw.max()
        bm25_norm = bm25_raw / bm25_max if bm25_max > 0 else np.zeros_like(bm25_raw)
        q_emb = PUBLIC_EMB_MODEL.encode([case_query], normalize_embeddings=True, show_progress_bar=False)
        dense_scores = (EXPANDED_EMBEDDINGS @ q_emb.T).flatten()
        scores = bm25_weight * bm25_norm + bge_weight * dense_scores
        top_idx = np.argsort(scores)[::-1]
        result = []
        for i in top_idx:
            if scores[i] <= 0:
                continue
            if case_id is not None and expanded_cases[i]["case_id"] == case_id:
                continue
            if len(result) >= top_k:
                break
            result.append({
                "case_id": expanded_cases[i]["case_id"],
                "verdict_label": expanded_cases[i]["verdict_label"],
                "score": round(float(scores[i]), 4),
            })
        return result

    print("\n=== Phase 3: Pseudo-Labeling Pass 2 (re-predict low-confidence) ===\n")
    pass2_results = {}
    for c in test_cases:
        cid = c["case_id"]
        if cid in HIGH_CONF_CASES:
            continue  # Already good
        cq = c["case_query"]

        # CBR on expanded set
        similar = find_similar_expanded(cq, case_id=cid)
        if similar:
            votes = defaultdict(float)
            for s in similar:
                votes[s["verdict_label"]] += s["score"]
            cbr_pred_p2 = max(votes, key=votes.get)
            cbr_weight_p2 = max(votes.values()) / sum(votes.values()) if sum(votes.values()) > 0 else 0.5
        else:
            cbr_pred_p2 = None
            cbr_weight_p2 = 0.0

        jp = JUDGE_PREDS.get(cid, {})
        j_ad = jp.get("adversarial", {})
        j_el = jp.get("elements_burden", {})
        j_re = jp.get("remedy_outcome", {})
        prim = PRIMARY_PREDS.get(cid, {})
        aud = AUDITOR_PREDS.get(cid, {})
        priv_pred_again, priv_sim, _ = private_semantic_cbr(cid, BGE_EMBEDDINGS, PRIMARY_PREDS, top_k=3)

        multi_votes = defaultdict(float)
        for w in TOP_K_WEIGHTS:
            pk, _ = ensemble_predict_7signals(
                prim.get("prediction"), prim.get("probabilities", {}), prim.get("confidence", "MEDIUM"),
                aud.get("prediction"), aud.get("confidence", "LOW"),
                cbr_pred_p2, cbr_weight_p2,
                priv_pred_again, priv_sim,
                None, 0.0,  # case_emb not available in pass2
                j_ad.get("label"), j_ad.get("confidence", "LOW"),
                j_el.get("label"), j_el.get("confidence", "LOW"),
                j_re.get("label"), j_re.get("confidence", "LOW"),
                w)
            multi_votes[pk] += 1
        pass2_results[cid] = max(multi_votes, key=multi_votes.get)

    print(f"Pass 2: {len(pass2_results)} cases re-predicted")
else:
    print("No high-confidence cases found, skipping pass 2")
    pass2_results = {}

print(f"Phase 3 total time: {time.time()-t_p3:.1f}s")


---
## MAIN: PREDICT ALL 60 PRIVATE CASES

In [ ]:
# ===========================================================================
# MAIN: PREDICT ALL 60 PRIVATE CASES
# ===========================================================================
print("\n=== Predicting private cases ===\n")
t_start = time.time()
results = []
pred_dist = Counter()

for idx, c in enumerate(test_cases):
    cid = c["case_id"]
    cq = c["case_query"]

    # Phase 3: If pass1/2 already predicted this case
    if cid in HIGH_CONF_CASES:
        pred = HIGH_CONF_CASES[cid]
    elif cid in pass2_results:
        pred = pass2_results[cid]
    else:
        pred = None  # run full ensemble

    if pred is not None:
        pred_dist[pred] += 1
        ev_entry = EVIDENCE_CACHE.get(cid, {})
        chunk_ids = ev_entry.get("chunk_ids", [])
        law_evidence = get_law_evidence_fused(cid, top_k=TOP_K_LAWS) if "get_law_evidence_fused" in dir() else PRECOMPUTED_LAW.get(cid, [])
        results.append({
            "case_id": cid, "prediction": pred,
            "case_evidence": chunk_ids,
            "law_evidence": [{"law_id": l["law_id"], "aid": l["aid"]} for l in law_evidence[:TOP_K_LAWS]],
        })
        print("[%d/60] %s: %s [PSEUDO]" % (idx + 1, cid, pred))
        continue

    ev_entry = EVIDENCE_CACHE.get(cid, {})
    chunk_ids = ev_entry.get("chunk_ids", [])
    chunk_texts = ev_entry.get("chunk_texts", [])

    # CBR với Hybrid Search
    cbr_pred, cbr_similar, _ = predict_by_cbr_weighted(cq, case_id=cid)
    cbr_score = cbr_similar[0]["score"] if cbr_similar else None
    cbr_weight = cbr_score if cbr_score is not None else 0.5

    # Private Semantic CBR
    priv_cbr_pred, priv_cbr_maxsim, _ = private_semantic_cbr(cid, BGE_EMBEDDINGS, PRIMARY_PREDS, top_k=3)
    case_emb_pred, case_emb_maxsim, _ = private_semantic_cbr_case(cid, CASE_EMBEDDINGS, PRIMARY_PREDS, top_k=3)
    case_emb_weight = case_emb_maxsim if case_emb_maxsim > 0 else 0.0
    priv_cbr_weight = priv_cbr_maxsim if priv_cbr_maxsim > 0 else 0.0

    # Judge Pass predictions
    jp = JUDGE_PREDS.get(cid, {})
    j_ad = jp.get("adversarial", {})
    j_el = jp.get("elements_burden", {})
    j_re = jp.get("remedy_outcome", {})

    law_evidence = get_law_evidence_fused(cid, top_k=TOP_K_LAWS)

    prim = PRIMARY_PREDS.get(cid, {})
    primary_pred = prim.get("prediction")
    primary_probs = prim.get("probabilities", {})

    aud = AUDITOR_PREDS.get(cid, {})
    auditor_pred = aud.get("prediction")
    auditor_conf = aud.get("confidence", "LOW")

    has_amounts = bool(extract_all_amounts(" ".join(chunk_texts)))
    quant_hint = get_quantitative_hint_improved(chunk_texts) if has_amounts else ""

    # Ensemble Multi-Weights Majority Voting (7-signal)
    multi_scores = defaultdict(float)
    for w in TOP_K_WEIGHTS:
        _, scores_k = ensemble_predict_7signals(
            primary_pred, primary_probs, prim.get("confidence", "MEDIUM"),
            auditor_pred, auditor_conf,
            cbr_pred, cbr_weight,
            priv_cbr_pred, priv_cbr_weight,
            case_emb_pred, case_emb_weight,
            j_ad.get("label"), j_ad.get("confidence", "LOW"),
            j_el.get("label"), j_el.get("confidence", "LOW"),
            j_re.get("label"), j_re.get("confidence", "LOW"),
            w)
        for label, score in scores_k.items():
            multi_scores[label] += score
    pred = max(multi_scores, key=multi_scores.get)

    cbr_agree = (cbr_pred == pred) if cbr_pred else False

    # Calibration
    feats = extract_features_improved(
        primary_probs, primary_pred, auditor_conf,
        cbr_pred, cbr_agree, cbr_score, has_amounts,
        primary_confidence=prim.get("confidence"))
    cal_pred = apply_calibration_improved(feats, threshold=0.5)
    if cal_pred is not None:
        pred = cal_pred

    pred_dist[pred] += 1
    results.append({
        "case_id": cid,
        "prediction": pred,
        "case_evidence": chunk_ids,
        "law_evidence": [{"law_id": l["law_id"], "aid": l["aid"]} for l in law_evidence[:TOP_K_LAWS]],
    })

    extras = []
    if cbr_pred:
        extras.append("CBR=%s" % cbr_pred)
    if cal_pred:
        extras.append("CAL")
    print("[%d/60] %s: %s%s" % (idx + 1, cid, pred,
                                " [%s]" % ",".join(extras) if extras else ""))

# Save
submission = [{"case_id": r["case_id"], "prediction": r["prediction"],
               "case_evidence": r["case_evidence"],
               "law_evidence": r["law_evidence"]} for r in results]
with open(OUTPUT, "w", encoding="utf-8") as f:
    json.dump(submission, f, ensure_ascii=False, indent=2)
print("\nSaved: %s" % OUTPUT)


In [ ]:
# ===========================================================================
# VALIDATE
# ===========================================================================
print("\n=== Validation ===")
corpus_ids = set(law["law_id"] for law in laws)
test_ids_set = set(c["case_id"] for c in test_cases)
errors = []
seen = set()
for item in submission:
    cid = item["case_id"]
    if cid in seen:
        errors.append("Duplicate: %s" % cid)
    seen.add(cid)
    if cid not in test_ids_set:
        errors.append("Unknown: %s" % cid)
    if item["prediction"] not in VALID_LABELS:
        errors.append("Label: %s -> %s" % (cid, item["prediction"]))
    for ch in item["case_evidence"]:
        if not isinstance(ch, str) or not ch.startswith(cid):
            errors.append("Bad evidence: %s" % ch)
    for le in item["law_evidence"]:
        if le["law_id"] not in corpus_ids:
            errors.append("Law: %s" % le["law_id"])
        if not isinstance(le["aid"], int):
            errors.append("aid type: %s" % type(le["aid"]).__name__)
if errors:
    print("Errors (%d):" % len(errors))
    for e in errors[:10]:
        print("  %s" % e)
else:
    print("Valid: %d cases, format OK" % len(submission))
    print("Predictions:")
    for k, v in sorted(pred_dist.items()):
        print("  %s: %d" % (k, v))
    print("\nTotal time: %.0fs" % (time.time() - t_start))


In [ ]:
# ===========================================================================
# SO SÁNH CÁC CẢI TIẾN
# ===========================================================================
print("\n" + "=" * 60)
print("SO SÁNH CÁC CẢI TIẾN SO VỚI BASELINE")
print("=" * 60)
changes = [
    ("1. CBR Public",
     "50 public cases",
     "50 public + pseudo-labeled private cases"),
    ("2. CBR Private",
     "Không dùng",
     "Private-to-private BGE semantic similarity"),
    ("3. Ensemble Signals",
     "3 signals",
     "7 signals (+Judge passes + Private CBR)"),
    ("4. Law Retrieval",
     "Static law_aids",
     "Fused candidates (BM25+BGE+Qwen fusion)"),
    ("5. Semi-Supervised",
     "Không dùng",
     "Pseudo-label high-confidence cases → expand CBR base"),
    ("6. Ensemble Strategy",
     "Single best weights",
     "Top-5 weights majority vote"),
    ("7. Calibration",
     "LogisticRegression",
     "RandomForest / GradientBoosting (CV)"),
]
for title, old, new in changes:
    print("\n%s" % title)
    print("  Baseline: %s" % old)
    print("  Improved: %s" % new)

print("\n" + "-" * 60)
print("Top-%d ensemble weights:" % len(TOP_K_WEIGHTS))
if TOP_K_WEIGHTS:
    for i, w in enumerate(TOP_K_WEIGHTS):
        print("  [%d] %s" % (i+1, json.dumps(w)))
print("-" * 60)
